# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore the dataset structure: record sets and fields
record_sets = list(dataset.metadata.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    fields = rs.get('fields', [])
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    print(f"  Fields:")
    for field in fields:
        print(f"    - {field['@id']} ({field.get('name', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Extract data from each record set
selected_record_sets = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in selected_record_sets:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records.")
        if not df.empty:
            print(f"  Columns: {df.columns.tolist()}")
            display(df.head())
        print()
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a record set containing numeric data for analysis
from IPython.display import display

# For demonstration, pick the first non-empty dataframe
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        main_df = df
        break
else:
    main_record_set_id = None
    main_df = pd.DataFrame()

if main_record_set_id:
    print(f"Using record set: {main_record_set_id}\n")

    # Show all column names
    print("Available fields/columns:", main_df.columns.tolist())

    # Try to find a numeric field (float or int)
    numeric_cols = main_df.select_dtypes(include=[float, int]).columns.tolist()
    if len(numeric_cols) == 0:
        print("No numeric columns found for EDA.")
    else:
        numeric_field = numeric_cols[0]  # Use the first numeric column
        print(f"Selected numeric_field for filtering: {numeric_field}")

        # Example: Filter records based on a threshold
        threshold = main_df[numeric_field].mean()  # Use mean as example threshold
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical field (if one exists)
        cat_cols = main_df.select_dtypes(include=["object"]).columns.tolist()
        if cat_cols:
            group_field = cat_cols[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
else:
    print("No suitable record set with data available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(numeric_cols) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouping was possible, show a grouped barplot
    if cat_cols:
        group_field = cat_cols[0]
        plt.figure(figsize=(10,4))
        sns.barplot(
            data=main_df,
            x=group_field,
            y=numeric_field,
            errorbar=None
        )
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Average {numeric_field} grouped by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available or record set with data for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*The FAIR² dataset was loaded using the Croissant schema via the `mlcroissant` library. We reviewed available record sets and fields (referenced by their `@id`s), loaded data into Pandas DataFrames, performed basic filtering and normalization on numeric fields, and visualized record distributions. For more advanced analysis, consult the Croissant schema for field semantics and refer to the provided documentation for domain-specific interpretation. The exploratory code structure is reusable for similar FAIR-compliant datasets.*